In [6]:
from langchain.graphs import Neo4jGraph
from langchain.chains import GraphQAChain
from langchain.chat_models import ChatOpenAI

graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="dengzhijie250",
    driver_config={"database": None}
)

llm = ChatOpenAI()

chain = GraphQAChain.from_llm(llm, graph=graph)
result = chain.run("尹林一案件的时间和法院信息是什么？")
print(result)

ConfigurationError: Database name parameter for selecting database is not supported in Bolt Protocol Version(3, 0). Database name 'neo4j'.

In [8]:
from py2neo import Graph
import requests
import json

# ChatGPT API（chatanywhere）调用函数
def gpt_4_call(text, api_key, url="https://api.chatanywhere.tech/v1/chat/completions"):
    payload = json.dumps({
        "model": "gpt-4o-2024-11-20",
        "temperature": 1.0,
        "messages": [
            {"role": "system", "content": "你是一个专业的中文知识图谱助手，会根据图谱中的实体和关系，帮用户生成简洁、自然的中文描述。"},
            {"role": "user", "content": text}
        ]
    })
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json'
    }

    response = requests.post(url, headers=headers, data=payload)
    if response.status_code == 200:
        try:
            result = response.json()
            return result["choices"][0]["message"]["content"]
        except KeyError:
            return "Unexpected response format."
    else:
        return f"Error: {response.status_code} - {response.text}"

# 初始化 Neo4j
graph = Graph("bolt://localhost:7687", auth=("neo4j", "dengzhijie250"))

# 用户输入问题（即关键词）
user_input = "尹林"
API_KEY = "sk-MYqQWPnGTdVG08zQActaJFUwCXETOWBpVVVou4nnoiFB83W6"

# 查询 Neo4j：查找中心实体的一跳邻居关系
def query_graph(keyword, limit=10):
    cypher = f"""
    MATCH (center)
    WHERE center.name CONTAINS '{keyword}'
    WITH center LIMIT 1
    MATCH (center)-[r]-(neighbor)
    RETURN center.name AS center, type(r) AS relation, neighbor.name AS neighbor, labels(neighbor) AS labels
    LIMIT {limit}
    """
    try:
        return graph.run(cypher).data()
    except Exception as e:
        return {"error": str(e)}

# 构造给 GPT 的输入 prompt
def generate_prompt(keyword, result):
    return f"""
用户查询的是：{keyword}
下面是该实体在知识图谱中相关的一些信息（三元组）：

{json.dumps(result, indent=2, ensure_ascii=False)}

请你根据这些信息，用中文写一段简洁自然的描述，帮助用户了解这个实体的基本情况。
"""

# 主逻辑
result = query_graph(user_input)

if isinstance(result, dict) and "error" in result:
    print("❌ 查询出错：", result["error"])
else:
    prompt = generate_prompt(user_input, result)
    final_answer = gpt_4_call(prompt, api_key=API_KEY)
    print("✅ 最终回答：\n", final_answer)

✅ 最终回答：
 尹林与戴文凯因涉嫌容留他人吸毒受到起诉，案件经湖南省衡阳市蒸湘区人民法院一审审理。该案由衡阳市蒸湘区人民检察院提起公诉，案号为（2021）湘0408刑初198号，审理程序为刑事一审。裁判日期为2021年6月9日，根据《中华人民共和国刑法（1997年）》相关条款以及相关法律规定进行判决，案件发生地为湖南省衡阳市。
